# TokensToTranslation — English → German Translator

A sequence-to-sequence LSTM translator built from scratch in **PyTorch**.

No HuggingFace pipelines, no pretrained translation models — the encoder, decoder and the seq2seq loop are implemented by hand. Only generic layers (`nn.Embedding`, `nn.LSTM`, `nn.Linear`, `CrossEntropyLoss`, `Adam`) are used.

Final project for the Deep Learning Summer of Code. Run the cells top to bottom in Google Colab — nothing else needed.

**Pipeline:** download data → clean & tokenize → build vocab → Dataset/DataLoader → Encoder/Decoder/Seq2Seq → train → evaluate → translate → save/load.

## 1. Imports & device

Colab already has PyTorch, so there is nothing to `pip install`. We just pick the GPU if one is available.

In [ ]:
import os
import re
import random
import zipfile
import urllib.request

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## 2. Download the dataset

We use the [ManyThings.org](https://www.manythings.org/anki/) Anki `deu-eng` corpus. The site blocks the default urllib user-agent, so we spoof a browser header before downloading.

In [ ]:
DATA_URL = 'https://www.manythings.org/anki/deu-eng.zip'


def download_dataset(data_dir='data'):
    """Download + unzip the corpus if needed, return path to deu.txt."""
    os.makedirs(data_dir, exist_ok=True)
    zip_path = os.path.join(data_dir, 'deu-eng.zip')
    txt_path = os.path.join(data_dir, 'deu.txt')

    if os.path.exists(txt_path):
        print('Already downloaded:', txt_path)
        return txt_path

    print('Downloading ...')
    # manythings.org needs full browser-like headers, else it returns 406
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.manythings.org/anki/',
    }
    req = urllib.request.Request(DATA_URL, headers=headers)
    with urllib.request.urlopen(req) as resp, open(zip_path, 'wb') as out:
        out.write(resp.read())

    print('Extracting ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_dir)
    return txt_path


txt_path = download_dataset()
print('Ready:', txt_path)

## 3. Cleaning & tokenization

Plain whitespace tokenization. We lower-case, isolate punctuation so `hello!` becomes two tokens, and keep German umlauts (`äöüß`).

In [ ]:
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<sos>', '<eos>', '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]


def normalize(text):
    """Lower-case, trim, space out punctuation, drop odd characters."""
    text = text.lower().strip()
    text = re.sub(r'([.!?,;:])', r' \1 ', text)
    text = re.sub(r"[^a-z\u00e4\u00f6\u00fc\u00df.!?,;:']+", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize(text):
    """Whitespace tokenizer (text should already be normalised)."""
    return text.split()


# quick sanity check
print(normalize('I love India!'))
print(tokenize(normalize('I love India!')))

## 4. Load sentence pairs

The file is `english<TAB>german<TAB>attribution` per line — we keep the first two columns. Using the first ~5000 pairs keeps training fast.

In [ ]:
NUM_PAIRS = 5000


def load_pairs(path, num_pairs=NUM_PAIRS):
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.split('\t')
            if len(parts) < 2:
                continue
            eng, deu = normalize(parts[0]), normalize(parts[1])
            if eng and deu:
                pairs.append((eng, deu))
            if len(pairs) >= num_pairs:
                break
    return pairs


pairs = load_pairs(txt_path)
print(f'Loaded {len(pairs)} pairs')
for eng, deu in pairs[:5]:
    print(f'  EN: {eng!r}  ->  DE: {deu!r}')

## 5. Vocabulary

Built by hand. Special tokens get indices 0–3 (so `<pad>` is 0), then every word seen at least `min_freq` times is added. We build one vocab for English and one for German.

In [ ]:
class Vocabulary:
    """Maps tokens <-> ids for a single language."""

    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}
        self._freq = {}
        for tok in SPECIAL_TOKENS:
            self._add_token(tok)

    def _add_token(self, token):
        if token not in self.word2idx:
            idx = len(self.word2idx)
            self.word2idx[token] = idx
            self.idx2word[idx] = token

    def build(self, sentences):
        for sentence in sentences:
            for token in tokenize(sentence):
                self._freq[token] = self._freq.get(token, 0) + 1
        for token, freq in self._freq.items():
            if freq >= self.min_freq:
                self._add_token(token)
        return self

    def numericalize(self, sentence):
        unk = self.word2idx[UNK_TOKEN]
        return [self.word2idx.get(t, unk) for t in tokenize(sentence)]

    def decode(self, ids):
        skip = {self.word2idx[t] for t in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN)}
        return [self.idx2word[i] for i in ids if i not in skip]

    def __len__(self):
        return len(self.word2idx)


src_vocab = Vocabulary(min_freq=1).build([p[0] for p in pairs])
trg_vocab = Vocabulary(min_freq=1).build([p[1] for p in pairs])
print('English vocab size:', len(src_vocab))
print('German  vocab size:', len(trg_vocab))

## 6. Dataset & DataLoader

The `Dataset` numericalizes every pair and wraps it with `<sos> ... <eos>`. The `collate_fn` pads each batch to its own longest sequence. Batch size 64, shuffled.

In [ ]:
MAX_LEN = 20
BATCH_SIZE = 64
PAD_IDX = src_vocab.word2idx[PAD_TOKEN]


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab, max_len=MAX_LEN):
        self.max_len = max_len
        self.sos = trg_vocab.word2idx[SOS_TOKEN]
        self.eos = trg_vocab.word2idx[EOS_TOKEN]
        self.data = []
        for eng, deu in pairs:
            self.data.append((
                self._encode(eng, src_vocab),
                self._encode(deu, trg_vocab),
            ))

    def _encode(self, sentence, vocab):
        ids = vocab.numericalize(sentence)[: self.max_len - 2]
        return [self.sos] + ids + [self.eos]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, trg = self.data[idx]
        return (torch.tensor(src, dtype=torch.long),
                torch.tensor(trg, dtype=torch.long))


def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_max = max(len(s) for s in src_batch)
    trg_max = max(len(t) for t in trg_batch)

    def pad(seq, length):
        extra = torch.full((length - len(seq),), PAD_IDX, dtype=torch.long)
        return torch.cat([seq, extra])

    src_padded = torch.stack([pad(s, src_max) for s in src_batch])
    trg_padded = torch.stack([pad(t, trg_max) for t in trg_batch])
    return src_padded, trg_padded


dataset = TranslationDataset(pairs, src_vocab, trg_vocab)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

sample_src, sample_trg = next(iter(loader))
print('src batch:', sample_src.shape, '| trg batch:', sample_trg.shape)

## 7. Model — Encoder

Embeds the English tokens, runs them through an LSTM and returns only the final `(hidden, cell)` — that compressed context is the bridge to the decoder. Everything is `batch_first=True`.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(
            emb_dim, hidden_dim, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [batch, src_len]
        embedded = self.dropout(self.embedding(src))
        _, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

## 8. Model — Decoder

Generates one German token per call. Given the previous token id and the current `(hidden, cell)`, it returns logits over the German vocabulary plus the updated state. The Seq2Seq module drives the loop.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(
            emb_dim, hidden_dim, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell):
        # input_token: [batch] -> add a time dimension
        input_token = input_token.unsqueeze(1)          # [batch, 1]
        embedded = self.dropout(self.embedding(input_token))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))     # [batch, output_dim]
        return prediction, hidden, cell

## 9. Model — Seq2Seq (with teacher forcing)

Wires encoder + decoder together. At each step we flip a coin against `teacher_forcing_ratio`: heads → feed the real previous token, tails → feed the model's own guess. Set the ratio to 0 for pure inference behaviour.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=self.device)

        hidden, cell = self.encoder(src)
        input_token = trg[:, 0]  # <sos> for every sequence

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = trg[:, t] if teacher_force else top1

        return outputs

## 10. Hyper-parameters & model instantiation

In [ ]:
EMB_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.2
LEARNING_RATE = 0.001
EPOCHS = 10
CLIP = 1.0
TEACHER_FORCING_RATIO = 0.5

encoder = Encoder(len(src_vocab), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
decoder = Decoder(len(trg_vocab), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {n_params:,}')

## 11. Training loop

`Adam` optimizer, `CrossEntropyLoss` that ignores padding, and gradient clipping to keep the LSTM stable. We drop the `<sos>` column before computing the loss and flatten to `[batch*time, vocab]`.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def train_one_epoch(model, loader, optimizer, criterion, clip, tf_ratio):
    model.train()
    epoch_loss = 0.0
    for src, trg in loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio=tf_ratio)

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg_flat = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)


print(f'Training on {DEVICE} for {EPOCHS} epochs\n')
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, loader, optimizer, criterion, CLIP, TEACHER_FORCING_RATIO)
    print(f'Epoch {epoch:02d}/{EPOCHS} | loss {loss:.4f}')

## 12. Evaluation

Same forward pass but with teacher forcing switched **off** (`ratio=0`), so the reported loss reflects the model feeding itself — closer to real inference. Here we just re-use the training set for a quick sanity number; with more data you'd hold out a validation split.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0.0
    for src, trg in loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        output = model(src, trg, teacher_forcing_ratio=0.0)
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg_flat = trg[:, 1:].reshape(-1)
        epoch_loss += criterion(output, trg_flat).item()
    return epoch_loss / len(loader)


val_loss = evaluate(model, loader, criterion)
print(f'Eval loss (no teacher forcing): {val_loss:.4f}')

## 13. Translation

Greedy decoding: encode the English sentence, start the decoder from `<sos>`, and keep feeding its own prediction back in until it emits `<eos>` (or we hit `max_len`).

In [ ]:
@torch.no_grad()
def translate_sentence(sentence, model=model, src_vocab=src_vocab,
                       trg_vocab=trg_vocab, max_len=MAX_LEN):
    """Translate one English sentence into German (greedy)."""
    model.eval()

    sos_s = src_vocab.word2idx[SOS_TOKEN]
    eos_s = src_vocab.word2idx[EOS_TOKEN]
    src_ids = [sos_s] + src_vocab.numericalize(normalize(sentence)) + [eos_s]
    src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    hidden, cell = model.encoder(src_tensor)

    trg_sos = trg_vocab.word2idx[SOS_TOKEN]
    trg_eos = trg_vocab.word2idx[EOS_TOKEN]
    input_token = torch.tensor([trg_sos], dtype=torch.long, device=DEVICE)

    out_ids = []
    for _ in range(max_len):
        output, hidden, cell = model.decoder(input_token, hidden, cell)
        pred = output.argmax(1)
        if pred.item() == trg_eos:
            break
        out_ids.append(pred.item())
        input_token = pred

    return ' '.join(trg_vocab.decode(out_ids))


print(translate_sentence('I love India.'))

## 14. Demo translations

At least five English sentences translated into German.

In [ ]:
demo_sentences = [
    'I love India.',
    'She is reading a book.',
    'We are going home.',
    'The weather is nice today.',
    'He drinks coffee every morning.',
    'Tom is my friend.',
]

for s in demo_sentences:
    print(f'EN: {s}')
    print(f'DE: {translate_sentence(s)}\n')

## 15. Save the model

We store the weights **and** both vocabularies in a single checkpoint so loading later needs nothing else.

In [ ]:
MODEL_PATH = 'translator.pt'

torch.save({
    'model_state': model.state_dict(),
    'src_word2idx': src_vocab.word2idx,
    'src_idx2word': src_vocab.idx2word,
    'trg_word2idx': trg_vocab.word2idx,
    'trg_idx2word': trg_vocab.idx2word,
    'config': {
        'emb_dim': EMB_DIM, 'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS, 'dropout': DROPOUT, 'max_len': MAX_LEN,
    },
}, MODEL_PATH)
print('Saved ->', MODEL_PATH)

## 16. Load the model

Rebuild the vocabularies and the network from the checkpoint, then translate to confirm the reloaded model behaves identically.

In [ ]:
def rebuild_vocab(word2idx, idx2word):
    v = Vocabulary()
    v.word2idx = word2idx
    v.idx2word = {int(k): val for k, val in idx2word.items()}
    return v


def load_model(path=MODEL_PATH):
    ckpt = torch.load(path, map_location=DEVICE)
    cfg = ckpt['config']
    sv = rebuild_vocab(ckpt['src_word2idx'], ckpt['src_idx2word'])
    tv = rebuild_vocab(ckpt['trg_word2idx'], ckpt['trg_idx2word'])

    enc = Encoder(len(sv), cfg['emb_dim'], cfg['hidden_dim'], cfg['num_layers'], cfg['dropout'])
    dec = Decoder(len(tv), cfg['emb_dim'], cfg['hidden_dim'], cfg['num_layers'], cfg['dropout'])
    m = Seq2Seq(enc, dec, DEVICE).to(DEVICE)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    return m, sv, tv, cfg['max_len']


loaded_model, loaded_src, loaded_trg, loaded_max_len = load_model()
print('Reloaded model. Test translation:')
print('EN: I love India.')
print('DE:', translate_sentence('I love India.', loaded_model, loaded_src, loaded_trg, loaded_max_len))

## Wrap-up

That's the full pipeline: a from-scratch seq2seq LSTM translator, trained on 5k English/German pairs, with save/load and greedy decoding.

With such a small corpus and 10 epochs it nails common short sentences and gets fuzzier on rarer vocabulary. The obvious next step (and the natural continuation of the SoC syllabus) is to bolt an **attention** layer onto the decoder, then try **beam search** and the **full** dataset.